# Dual-Path Multi-Source Visual Search Engine (MSSearch) on 2x NVIDIA T4 GPUs
## Full Benchmark: HRBench 8K with Fail-Safe Incremental Checkpointing

> **Objective**: Execute the complete **Dual-Path Multi-Source Visual Search Engine (`MSSearch`)** on **2x NVIDIA T4 GPUs (16GB VRAM each = 32GB total)** using **pure `float16` precision (ZERO 4-bit quantization)** across the entire **HRBench 8K** dataset.

### 🛡️ Fail-Safe 12-Hour Session Persistence & Auto-Resume
- **Per-Sample Incremental Streaming**: Every single prediction is appended immediately to `/kaggle/working/answers/mssearch_hrbench8k_predictions.jsonl`.
- **Physical Disk Flushing**: Uses `f.flush()` and `os.fsync(f.fileno())` so crashes, kernel interruptions, or Kaggle 12-hour session timeouts retain 100% of finished work.
- **Automatic Crash Recovery**: On notebook start or restart, existing lines are scanned and already-completed sample indices are skipped without redundant computation.

### 🔍 Dynamic Semantic Partitioning (Paradigm 2 Saliency Plateau)
- **Self-Terminating Zoom**: Replaces the rigid `max_patch_size = 512` recursive bounding constraint with dynamic semantic saliency gain ($\Delta S \le 0.04$).
- **Zero Target Fragmentation**: Medium and large targets ($> 512	ext{px}$) are kept intact without being shattered across quadrant borders, preventing refusal hallucinations.
- **Dual Mode Switch**: Configurable via `partition_stopping_mode="saliency_plateau"` vs `"fixed_size"`.

### ⚡ Dual-GPU Memory Distribution
- **GPU 0 (`cuda:0`)**: Qwen2.5-VL Vision Tower + Layers 0-14 + SigLIP-base-patch16-384 (~8.7 GB VRAM, ~6.7 GB headroom).
- **GPU 1 (`cuda:1`)**: Qwen2.5-VL Layers 15-27 + LM Head + SAM3 Image Foundation Model (~11.7 GB VRAM, ~3.7 GB headroom).


In [ ]:
import os
import sys
import gc
import json
import time
import torch
from PIL import Image

print("=" * 75)
print("SYSTEM TELEMETRY & MULTI-GPU DIAGNOSTICS")
print("=" * 75)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"GPU Device Count: {device_count}")

if device_count < 2:
    print("[WARNING] Less than 2 GPUs detected. Operating in single-GPU mode.")
else:
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"  [GPU {i}] {props.name} | Compute: {props.major}.{props.minor} | VRAM: {vram_gb:.2f} GB")

has_sdpa = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
print(f"PyTorch SDPA Available: {has_sdpa}")
print("=" * 75)


In [ ]:
# ==============================================================================
# DATASET & MODEL PATHS CONFIGURATION (HRBench 8K Full Benchmark)
# ==============================================================================

# Foundation Model Weights
SAM3_CHECKPOINT_PATH = "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt"
QWEN_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"
SIGLIP_MODEL_NAME = "google/siglip-base-patch16-384"
NLP_MODEL_PATH = "en_core_web_sm"

# HRBench 8K Dataset Path Detection
HRBENCH_BASE_DIR = "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k"
HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_8k.json")

if not os.path.exists(HRBENCH_ANNOTATION_PATH):
    alt_paths = [
        "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/annotation_hr-bench_8k.json",
        "/kaggle/input/vs-dataset/hr-bench_8k/annotation_hr-bench_8k.json",
        "/kaggle/input/hr-bench-8k/annotation_hr-bench_8k.json"
    ]
    for alt in alt_paths:
        if os.path.exists(alt):
            HRBENCH_ANNOTATION_PATH = alt
            HRBENCH_BASE_DIR = os.path.dirname(alt)
            break

# Checkpointing & Output Knobs
OUTPUT_DIR = "/kaggle/working/answers"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_JSONL = os.path.join(OUTPUT_DIR, "mssearch_hrbench8k_predictions.jsonl")

# Execution Windowing (Set MAX_SAMPLES = None to process entire dataset)
START_INDEX = 0
MAX_SAMPLES = None
SMOKE_TEST_MODE = False        # Set True to quickly verify first 3 samples
DEBUG_VISUALIZE = False        # Inline visualization of crops (keep False for high-throughput headless runs)

print(f"[CONFIG] HRBench Annotation : {HRBENCH_ANNOTATION_PATH} (Exists: {os.path.exists(HRBENCH_ANNOTATION_PATH)})")
print(f"[CONFIG] HRBench Image Dir  : {HRBENCH_BASE_DIR}")
print(f"[CONFIG] SAM3 Weights       : {SAM3_CHECKPOINT_PATH} (Exists: {os.path.exists(SAM3_CHECKPOINT_PATH)})")
print(f"[CONFIG] Streaming Output   : {PREDICTIONS_JSONL}")


In [ ]:
# Clone or update MSSearch repository from GitHub using Kaggle Secret 'MaCVi'
import os
import sys
import subprocess
import importlib
import urllib.request

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    print(f"[SETUP] Note: UserSecretsClient notice ({e}). Checking environment variables...")
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        print(f"[SETUP] Cloning MSSearch repository using token into {REPO_DIR}...")
        clone_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        print(f"[SETUP] Attempting unauthenticated clone into {REPO_DIR}...")
        clone_url = "https://github.com/tothanhdat2006/MSSearch.git"
    ret = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Successfully cloned MSSearch into {REPO_DIR}")
    else:
        safe_err = ret.stderr.replace(github_token, "******") if github_token else ret.stderr
        print(f"[SETUP] Git clone failed: {safe_err}")
else:
    print(f"[SETUP] MSSearch repository already exists at {REPO_DIR}. Pulling latest changes from main...")
    ret = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Repository updated to latest: {ret.stdout.strip()}")
    else:
        safe_err = ret.stderr.replace(github_token, "******") if github_token else ret.stderr
        print(f"[SETUP] Git pull note: {safe_err.strip()}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('mssearch'):
        try:
            importlib.reload(sys.modules[mod_name])
        except Exception:
            pass

bpe_target = os.path.join(REPO_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = "https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz"
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print("[SETUP] Downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download: {e}")

!pip install -q scikit-image scikit-learn networkx iopath decord ftfy

from mssearch.config import MSSearchConfig
from mssearch.MSSearch import MultiSourceSearchEngine, get_mssearch_response
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_sam3 import sam3_inference
from mssearch.models.modeling_siglip import SigLIPScorer

print("[SUCCESS] Successfully imported MSSearch core engine and model adapters!")


In [ ]:
print("=" * 75)
print("INITIALIZING FOUNDATION MODELS & DUAL-GPU ASYMMETRIC DISTRIBUTION")
print("=" * 75)

# 1. Load Qwen2.5-VL in float16 (Sharded across GPU 0 & GPU 1)
print("[1/4] Loading Qwen2.5-VL-7B-Instruct (Pure float16 with SDPA backend)...")
qwen_engine = ModelQwenVL(
    model_path=QWEN_MODEL_PATH,
    torch_dtype=torch.float16,
    patch_scale=1.2
)

# 2. Load SAM3 on cuda:1 (5.2GB allocation alongside Qwen layers 15-27)
sam3_device = "cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0"
print(f"[2/4] Loading SAM3 image model on {sam3_device}...")
sam3_engine = sam3_inference(
    model_path=SAM3_CHECKPOINT_PATH,
    device=sam3_device
)

# 3. Load SigLIP-base on cuda:0 (Dense spatial token scorer)
siglip_device = "cuda:0"
print(f"[3/4] Loading SigLIP Scorer ({SIGLIP_MODEL_NAME}) on {siglip_device}...")
siglip_engine = SigLIPScorer(
    model_name=SIGLIP_MODEL_NAME,
    device=siglip_device,
    torch_dtype=torch.float16
)

# 4. Load SpaCy NLP Parser on CPU
print(f"[4/4] Loading SpaCy NLP parser ({NLP_MODEL_PATH})...")
import spacy
try:
    nlp_parser = spacy.load(NLP_MODEL_PATH)
except Exception:
    !python -m spacy download en_core_web_sm
    nlp_parser = spacy.load("en_core_web_sm")

print("-" * 75)
for i in range(torch.cuda.device_count()):
    alloc_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    res_gb = torch.cuda.memory_reserved(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  [GPU {i}] Allocated: {alloc_gb:.2f} GB | Reserved: {res_gb:.2f} GB | Total: {total_gb:.2f} GB | Headroom: {total_gb - res_gb:.2f} GB")
print("=" * 75)


In [ ]:
# Configure MSSearch for Full HRBench 8K Execution
config_hrbench = MSSearchConfig(
    enable_direct_answer=True,           # Phase 1: Direct Global Answer via logit sufficiency probing
    direct_answer_conf_threshold=0.60,   # Sufficiency threshold c_q in [-1, 1] on root view
    enable_fast_path=True,               # Phase 2: Tool-Based Fast Path via SAM3 full-canvas detection
    fast_path_conf_threshold=0.70,       # Existence confidence threshold for fast path instant exit
    enable_anchor_first_ordering=True,   # Prioritize physical host entities ahead of abstract attributes
    enable_query_adaptive_dual_scale=True,# Query-Adaptive Dual-Scale visual prompting for relational queries
    prompt_directed_final_answer=True,   # Enforce prompt-directed Final Answer format for clean extraction
    entropy_threshold=0.35,
    min_patch_entropy=0.35,              # Preserves smooth painted targets
    enable_latent_gating=True,
    latent_confidence_threshold=0.0,
    enable_existence_gating=True,
    existence_confidence_threshold=0.0,
    partition_mode="hybrid",
    partition_stopping_mode="saliency_plateau", # "saliency_plateau" (Paradigm 2 dynamic semantic zoom) or "fixed_size"
    max_patch_size=512,                  # High-resolution boundary standard (used when fixed_size)
    min_plateau_size=256,                # Scale floor for saliency plateau stopping
    max_plateau_size=1536,               # Physical scale ceiling (GPU safety)
    saliency_plateau_gain_threshold=0.04,# Minimum saliency gain required to descend deeper
    saliency_target_threshold=0.35,      # High confidence target threshold where marginal gain halts zoom
    max_patches_per_image=100,
    bbox_expansion_factor=1.25,
    enable_visual_prompt_bbox=True,
    max_sam3_eval_patches=16,            # Stage 4 prioritized scan cap
    max_vlm_tokens=128,                  # Response token budget (allows complete answers without truncation)
    early_consensus_count=2,             # Stage 6 early exit consensus count
    early_consensus_threshold=0.75,      # Collision-free clean final answer similarity threshold
    verbose=False                        # Clean high-throughput output
)

print("[CONFIG] MSSearch optimized parameters successfully set for HRBench 8K (Paradigm 2 Saliency Plateau)!")


In [ ]:
print("=" * 75)
print("STARTING FAIL-SAFE INCREMENTAL BENCHMARK LOOP: HRBENCH 8K")
print("=" * 75)

# 1. Ingest Annotation Data
with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    hrbench_samples = json.load(f)

total_available = len(hrbench_samples)
print(f"[Dataset] Ingested {total_available} total items from {HRBENCH_ANNOTATION_PATH}")

# 2. Scan Existing Checkpoints for Resume Invariant
completed_indices = set()
if os.path.exists(PREDICTIONS_JSONL):
    with open(PREDICTIONS_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if 'index' in rec and rec['index'] is not None:
                    completed_indices.add(rec['index'])
            except Exception:
                pass
    print(f"[Resume] Found {len(completed_indices)} existing predictions in {PREDICTIONS_JSONL}.")

# Slice dataset according to execution knobs
if SMOKE_TEST_MODE:
    target_samples = hrbench_samples[:3]
else:
    end_idx = (START_INDEX + MAX_SAMPLES) if MAX_SAMPLES else total_available
    target_samples = hrbench_samples[START_INDEX:end_idx]

print(f"[Execution Window] Processing {len(target_samples)} items (Indices {START_INDEX} to {START_INDEX + len(target_samples) - 1})")

# 3. Execute Loop with Append-Only Physical Disk Flushing
session_processed = 0
total_latencies = []
loop_start_time = time.time()

with open(PREDICTIONS_JSONL, "a", encoding="utf-8") as out_f:
    for i, sample in enumerate(target_samples):
        sample_idx = sample.get('index', i)
        
        # Skip if already computed
        if sample_idx in completed_indices:
            continue
        
        img_rel = sample.get('input_image', '')
        img_full_path = os.path.join(HRBENCH_BASE_DIR, img_rel)
        
        if not os.path.exists(img_full_path):
            # Try direct relative check
            if os.path.exists(img_rel):
                img_full_path = img_rel
            else:
                print(f"[Sample {sample_idx}] Image missing: '{img_full_path}', recording missing entry.")
                missing_rec = {
                    "index": sample_idx,
                    "question": sample.get('question', ''),
                    "category": sample.get('category', 'single'),
                    "options": sample.get('options', {}),
                    "answer": sample.get('answer', ''),
                    "mssearch_prediction": "[ERROR: IMAGE NOT FOUND]",
                    "latency_sec": 0.0,
                    "route": "error_missing_image"
                }
                out_f.write(json.dumps(missing_rec, ensure_ascii=False) + "\n")
                out_f.flush()
                os.fsync(out_f.fileno())
                completed_indices.add(sample_idx)
                continue
        
        try:
            img_pil = Image.open(img_full_path).convert("RGB")
        except Exception as e:
            print(f"[Sample {sample_idx}] Failed to open image: {e}")
            continue
        
        t0 = time.perf_counter()
        try:
            response = get_mssearch_response(
                sam_model=sam3_engine,
                zoom_model=qwen_engine,
                siglip_model=siglip_engine,
                nlp_model=nlp_parser,
                annotation=sample,
                candidate_images=[img_pil],
                config=config_hrbench
            )
        except Exception as e:
            print(f"[Sample {sample_idx}] Runtime exception in get_mssearch_response: {e}")
            response = {
                'primary_answer': f'[ERROR: {str(e)}]',
                'answers': [],
                'telemetry': {'route': 'error_exception', 'timings_sec': {}}
            }
        
        latency = round(time.perf_counter() - t0, 3)
        total_latencies.append(latency)
        session_processed += 1
        
        pred_answer = response.get('primary_answer', '')
        gt_answer = sample.get('answer', sample.get('ground_truth', ''))
        cycle_idx = sample.get('cycle_category', 0)
        try:
            cycle_idx = int(cycle_idx)
        except Exception:
            cycle_idx = 0
        if isinstance(gt_answer, (list, tuple)) and 0 <= cycle_idx < len(gt_answer):
            gt_letter = str(gt_answer[cycle_idx]).strip().upper()
        else:
            gt_letter = str(gt_answer).strip().upper()
        telemetry = response.get('telemetry', {})
        route = telemetry.get('route', 'unknown')
        
        # Format checkpoint record with full audit metadata
        record = {
            "index": sample_idx,
            "question": sample.get('question', ''),
            "category": sample.get('category', 'single'),
            "cycle_category": cycle_idx,
            "options": sample.get('options', {}),
            "answer": gt_answer,
            "gt_letter": gt_letter,
            "input_image": img_rel,
            "mssearch_prediction": pred_answer,
            "raw_vlm_response": response.get("raw_vlm_response", ""),
            "extracted_answers": response.get('answers', []),
            "route": route,
            "latency_sec": latency,
            "evidence_images": response.get('evidence_images', []),
            "timings_sec": telemetry.get('timings_sec', {})
        }
        
        # Immediate Append-Only Physical Disk Flush
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        out_f.flush()
        os.fsync(out_f.fileno())
        completed_indices.add(sample_idx)
        
        # Rolling Telemetry Log
        avg_lat = sum(total_latencies) / len(total_latencies)
        remain_count = len(target_samples) - len(completed_indices)
        est_remaining_sec = avg_lat * max(0, remain_count)
        est_rem_hrs = est_remaining_sec / 3600.0
        
        pred_clean = pred_answer.replace('\n', ' ').strip()
        pred_disp = (pred_clean[:75] + '...') if len(pred_clean) > 75 else pred_clean
        print(f"[{len(completed_indices)}/{total_available}] Sample {sample_idx} ({sample.get('category', 'single')}) | "
              f"Pred: '{pred_disp}' | Route: {route} | Latency: {latency:.2f}s (Avg: {avg_lat:.2f}s | ETA: {est_rem_hrs:.2f}h)")
        
        # Garbage collection shield
        gc.collect()
        torch.cuda.empty_cache()

print("=" * 75)
print(f"BENCHMARK SESSION COMPLETE!")
print(f"  • Newly processed this session : {session_processed}")
print(f"  • Total completed on disk     : {len(completed_indices)} / {total_available}")
print(f"  • Output saved & synced to    : {PREDICTIONS_JSONL}")
print("=" * 75)


In [ ]:
print("=" * 75)
print("POST-FLIGHT TELEMETRY & ZERO-OOM MEMORY AUDIT")
print("=" * 75)

oom_safe = True
for i in range(torch.cuda.device_count()):
    max_alloc_gb = torch.cuda.max_memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    headroom = total_gb - max_alloc_gb
    print(f"[GPU {i}] Peak VRAM: {max_alloc_gb:.2f} GB / {total_gb:.2f} GB | Safety Margin: {headroom:.2f} GB")
    if max_alloc_gb > 15.0:
        oom_safe = False

print(f"\nZero-OOM Safety Condition Maintained: {oom_safe}")

# Summary count of completed lines
if os.path.exists(PREDICTIONS_JSONL):
    with open(PREDICTIONS_JSONL, "r", encoding="utf-8") as f:
        lines = [json.loads(line) for line in f if line.strip()]
    print(f"Total verified samples on disk: {len(lines)}")
    routes = [r.get('route', 'unknown') for r in lines]
    from collections import Counter
    print("Execution routes breakdown:", Counter(routes))
print("=" * 75)
